In [ ]:
# ── Cell 1: Mount Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
scVI_synthetic_generation_verified.py
=====================================
Complete, self-contained scVI synthetic-cell generation with a built-in
checkpoint sanity check.

Differences vs. the original "Copy of scVI_Synthetic_file_generation":
  1. The checkpoint path is auto-resolved, so the one-space / two-space
     folder-name mismatch can no longer silently send you to a non-existent dir.
  2. `.expect_partial()` is replaced by a strict load that PROVES the trained
     weights are in the model (file exists -> variables matched -> weights
     changed -> reconstruction tracks real data). If any check fails it raises,
     so you can never accidentally generate from random weights.
  3. Loops over BOTH datasets (PBMC, PDO) instead of hardcoding one.

Run top-to-bottom in Colab. Model classes are byte-for-byte the same as the
training notebook so the checkpoint variable names match.
"""

import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

EPS = 1e-8
SEED = 42

# ============================================================================
# 1. Model (verbatim — attribute names MUST match the checkpoint)
# ============================================================================
def log_zinb_positive(x, mu, theta, pi):
    if len(theta.shape) == 1:
        theta = tf.expand_dims(theta, 0)
    softplus_pi      = tf.nn.softplus(-pi)
    log_theta_eps    = tf.math.log(theta + EPS)
    log_theta_mu_eps = tf.math.log(theta + mu + EPS)
    pi_theta_log     = -pi + theta * (log_theta_eps - log_theta_mu_eps)
    case_zero     = tf.nn.softplus(pi_theta_log) - softplus_pi
    case_non_zero = (-softplus_pi + pi_theta_log
        + x * (tf.math.log(mu + EPS) - log_theta_mu_eps)
        + tf.math.lgamma(x + theta) - tf.math.lgamma(theta) - tf.math.lgamma(x + 1.0))
    mask = tf.cast(x < EPS, tf.float32)
    return mask * case_zero + (1.0 - mask) * case_non_zero


class FCBlock(keras.layers.Layer):
    def __init__(self, n_out, dropout_rate=0.1, use_bn=True, **kw):
        super().__init__(**kw)
        self.dense = keras.layers.Dense(n_out, use_bias=True)
        self.bn    = (keras.layers.BatchNormalization(momentum=0.99, epsilon=0.001) if use_bn else None)
        self.relu  = keras.layers.ReLU()
        self.drop  = keras.layers.Dropout(dropout_rate) if dropout_rate > 0 else None
    def call(self, x, training=False):
        x = self.dense(x)
        if self.bn:   x = self.bn(x, training=training)
        x = self.relu(x)
        if self.drop: x = self.drop(x, training=training)
        return x


class Encoder(keras.layers.Layer):
    def __init__(self, n_input, n_output, n_hidden=128, n_layers=1, dropout_rate=0.1, **kw):
        super().__init__(**kw)
        self.fc     = [FCBlock(n_hidden, dropout_rate, name=f'fc{i}') for i in range(n_layers)]
        self.mean_h = keras.layers.Dense(n_output, name='mean_head')
        self.var_h  = keras.layers.Dense(n_output, name='var_head')
    def call(self, x, training=False):
        h = x
        for fc in self.fc: h = fc(h, training=training)
        q_m = self.mean_h(h)
        q_v = tf.exp(self.var_h(h)) + 1e-4
        z   = q_m + tf.sqrt(q_v) * tf.random.normal(tf.shape(q_m))
        return q_m, q_v, z


class DecoderSCVI(keras.layers.Layer):
    def __init__(self, n_latent, n_output, n_hidden=128, n_layers=1, **kw):
        super().__init__(**kw)
        self.fc      = [FCBlock(n_hidden, dropout_rate=0.0, name=f'fc{i}') for i in range(n_layers)]
        self.scale_h = keras.layers.Dense(n_output, name='scale_head')
        self.drop_h  = keras.layers.Dense(n_output, name='dropout_head')
    def call(self, z, library, training=False):
        h = z
        for fc in self.fc: h = fc(h, training=training)
        px_scale   = tf.nn.softmax(self.scale_h(h), axis=-1)
        px_dropout = self.drop_h(h)
        px_rate    = tf.exp(library) * px_scale
        return px_scale, px_rate, px_dropout


class scVI(keras.Model):
    def __init__(self, n_input, n_hidden=128, n_latent=10, n_layers=1,
                 dropout_rate=0.1, dispersion='gene', **kw):
        super().__init__(**kw)
        self.n_latent = n_latent; self.n_input = n_input; self.dispersion = dispersion
        self.log_theta = tf.Variable(tf.random.normal([n_input]), trainable=True, name='log_theta')
        self.kl_weight = tf.Variable(0.0, trainable=False, dtype=tf.float32, name='kl_weight')
        self.z_encoder = Encoder(n_input, n_latent, n_hidden, n_layers, dropout_rate, name='z_encoder')
        self.l_encoder = Encoder(n_input, 1,        n_hidden, 1,         dropout_rate, name='l_encoder')
        self.decoder   = DecoderSCVI(n_latent, n_input, n_hidden, n_layers, name='decoder')
    def _encode(self, x, training=False):
        x_log = tf.math.log1p(x)
        qz_m, qz_v, z       = self.z_encoder(x_log, training=training)
        ql_m, ql_v, library = self.l_encoder(x_log, training=training)
        return qz_m, qz_v, z, ql_m, ql_v, library
    def _decode(self, z, library, training=False):
        px_scale, px_rate, px_dropout = self.decoder(z, library, training=training)
        return px_scale, px_rate, px_dropout, tf.exp(self.log_theta)
    def compute_elbo(self, x, local_l_mean, local_l_var, training=False):
        qz_m, qz_v, z, ql_m, ql_v, library = self._encode(x, training=training)
        _, px_rate, px_dropout, theta = self._decode(z, library, training=training)
        recon = -tf.reduce_sum(log_zinb_positive(x, px_rate, theta, px_dropout), axis=-1)
        return tf.reduce_mean(recon), 0, 0, 0   # forward pass only, to build vars


# ============================================================================
# 2. Config  (DRIVE_ROOT is auto-checked for the 1-space / 2-space variant)
# ============================================================================
# The training notebooks used a folder literally named "16. Review SynCellNet  work"
# (two spaces). We list both spellings and pick whichever exists on disk.
DRIVE_ROOT_CANDIDATES = [
    '/content/drive/MyDrive/Ahsan/16. Review SynCellNet  work',  # two spaces (training default)
    '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work',   # one space
]

def resolve_drive_root():
    for root in DRIVE_ROOT_CANDIDATES:
        if os.path.isdir(root):
            print(f"[path] using DRIVE_ROOT = {root!r}")
            return root
    raise FileNotFoundError(
        "Neither spacing variant of the Drive root exists. Candidates tried:\n  "
        + "\n  ".join(repr(r) for r in DRIVE_ROOT_CANDIDATES)
        + "\nList /content/drive/MyDrive/Ahsan to find the real folder name."
    )

def build_paths(root):
    return {
        'REAL_DATA': {
            'PBMC': {0: f'{root}/Dataset/Real PBMC dataset/b_Class_dataset.csv',
                     1: f'{root}/Dataset/Real PBMC dataset/mono_Class_dataset.csv'},
            'PDO':  {0: f'{root}/Dataset/Real PDO/3. Stem_High_Raw_Finalized.csv',
                     1: f'{root}/Dataset/Real PDO/3. Differential_Low_Raw_Finalized.csv'},
        },
        'CKPT_DIR': {
            'PBMC': f'{root}/scVI Approach/PBMC/checkpoints',
            'PDO':  f'{root}/scVI Approach/PDO/checkpoints',
        },
        'OUTPUT_DIR': {
            'PBMC': f'{root}/Dataset/Synthetic PBMC dataset/scVI/PBMC',
            'PDO':  f'{root}/Dataset/Synthetic PDO dataset/scVI/PDO',
        },
        'CLASS_NAMES': {
            'PBMC': {0: 'b', 1: 'mono'},
            'PDO':  {0: 'stem_high', 1: 'diff_low'},
        },
    }


# ============================================================================
# 3. Checkpoint sanity check  (replaces silent .expect_partial())
# ============================================================================
def verify_checkpoint_load(model, ckpt_dir, X_real=None, ckpt_name="best_model"):
    prefix = os.path.join(ckpt_dir, ckpt_name)
    print(f"[1/4] Locating checkpoint:\n      {prefix}")

    matches = glob.glob(prefix + "*")
    if not matches:
        listing = os.listdir(ckpt_dir) if os.path.isdir(ckpt_dir) else "<DIR MISSING>"
        raise FileNotFoundError(
            f"No checkpoint files matching {prefix!r}.\n"
            f"      Contents of ckpt_dir: {listing}"
        )
    print("      found:", [os.path.basename(m) for m in matches])

    # snapshot several trained weights before restore
    watched = {
        "decoder.scale_h": model.decoder.scale_h.kernel,
        "z_encoder.mean_h": model.z_encoder.mean_h.kernel,
        "log_theta": model.log_theta,
    }
    before = {k: v.numpy().copy() for k, v in watched.items()}

    print("[2/4] Restoring checkpoint…")
    # NB: training used ckpt.write() which stores no `save_counter`, so
    # assert_existing_objects_matched() would (harmlessly) trip on save_counter.
    # We use expect_partial() to silence that, then PROVE the load worked below
    # via the weight-change and correlation checks — those are the real guards.
    status = tf.train.Checkpoint(model=model).restore(prefix)
    status.expect_partial()

    after = {k: v.numpy() for k, v in watched.items()}
    print("[3/4] Weight change from restore:")
    n_changed = 0
    for k in watched:
        delta = float(np.abs(after[k] - before[k]).mean())
        changed = not np.allclose(before[k], after[k])
        n_changed += changed
        print(f"      {k:18s} mean|Δ| = {delta:.3e}  changed={changed}")
    assert n_changed == len(watched), (
        "Some weights did NOT change after restore — checkpoint not fully loaded."
    )
    print("      ✓ all watched weights changed → load took effect")

    if X_real is not None:
        _, _, z, _, _, lib = model._encode(tf.constant(X_real, tf.float32), training=False)
        _, px_rate, _, _ = model._decode(z, lib, training=False)
        corr = float(np.corrcoef(X_real.mean(0), px_rate.numpy().mean(0))[0, 1])
        print(f"[4/4] recon vs real per-gene-mean correlation: {corr:.4f}")
        # The weight-change check above already PROVES the trained checkpoint loaded.
        # This correlation is a FIT-QUALITY diagnostic, not a load test:
        #   ~0.0      -> random/untrained weights (catastrophic) -> hard fail
        #   0.1-0.5   -> correctly loaded but weak fit (honest "not great" result) -> warn
        #   >0.5      -> healthy fit
        assert corr > 0.1, (
            f"Correlation {corr:.3f} ~ 0: weights look random/untrained, not a real fit."
        )
        if corr < 0.5:
            print(f"      ⚠ WEAK FIT (corr={corr:.3f}): checkpoint IS loaded correctly "
                  f"(weights changed above), but the model reconstructs this dataset "
                  f"poorly. Consistent with weak clustering/imputation metrics. "
                  f"This is an honest result, not a load failure.")
        else:
            print("      ✓ reconstruction tracks the real data")
    else:
        print("[4/4] skipped functional check (no X_real passed)")

    print("ALL CHECKS PASSED — model is on the trained checkpoint.\n")
    return True


# ============================================================================
# 4. Class-conditional ZINB posterior-predictive generation
#    (gamma(shape=theta, scale=mu/theta) -> Poisson -> dropout : exact scVI sampler)
# ============================================================================
def generate_from_posterior(model, X_class, n_cells, seed):
    rng = np.random.default_rng(seed)
    qz_m, qz_v, _, ql_m, ql_v, _ = model._encode(tf.constant(X_class, tf.float32), training=False)
    qz_m, qz_v, ql_m, ql_v = qz_m.numpy(), qz_v.numpy(), ql_m.numpy(), ql_v.numpy()
    idx = rng.choice(len(X_class), size=n_cells, replace=(n_cells > len(X_class)))
    z_s = qz_m[idx] + np.sqrt(qz_v[idx]) * rng.standard_normal((n_cells, qz_m.shape[1]))
    l_s = ql_m[idx] + np.sqrt(ql_v[idx]) * rng.standard_normal((n_cells, 1))
    _, px_rate, px_dropout, theta = model._decode(
        tf.constant(z_s, tf.float32), tf.constant(l_s, tf.float32), training=False)
    mu, pi = px_rate.numpy(), px_dropout.numpy()
    th = np.broadcast_to(theta.numpy(), mu.shape)
    counts = rng.poisson(np.clip(rng.gamma(shape=th, scale=mu / (th + 1e-8)), 0, 1e8)).astype(np.float32)
    counts[rng.uniform(size=pi.shape) < 1.0 / (1.0 + np.exp(-pi))] = 0.0
    return counts


# ============================================================================
# 5. Driver — loops over both datasets
# ============================================================================
def run(datasets=('PBMC', 'PDO')):
    root  = resolve_drive_root()
    P     = build_paths(root)

    for DATASET in datasets:
        print("=" * 70)
        print(f"  DATASET = {DATASET}")
        print("=" * 70)

        # load real cells per class
        Xs = [pd.read_csv(P['REAL_DATA'][DATASET][k], header=None).values.astype(np.float32)
              for k in sorted(P['REAL_DATA'][DATASET])]
        n_genes = Xs[0].shape[1]

        # rebuild model + build all variables
        model = scVI(n_input=n_genes, n_hidden=128, n_latent=10, n_layers=1, dropout_rate=0.1)
        _ = model.compute_elbo(tf.zeros([2, n_genes]), tf.zeros([2, 1]), tf.ones([2, 1]))

        # >>> sanity check gates generation <<<
        verify_checkpoint_load(model, P['CKPT_DIR'][DATASET], X_real=Xs[0])

        # generate per class
        out_dir = P['OUTPUT_DIR'][DATASET]
        os.makedirs(out_dir, exist_ok=True)
        for k, name in P['CLASS_NAMES'][DATASET].items():
            syn = generate_from_posterior(model, Xs[k], len(Xs[k]), seed=SEED + k)
            fpath = os.path.join(out_dir, f'scvi_synthetic_{name}_expression_v3.csv')
            pd.DataFrame(syn).to_csv(fpath, header=False, index=False)
            print(f"  {name}: {syn.shape}  sparsity {(syn == 0).mean() * 100:.1f}%  ->  {fpath}")
        print()


if __name__ == "__main__":
    # In Colab, mount Drive first:
    #   from google.colab import drive; drive.mount('/content/drive')
    run(('PBMC', 'PDO'))


[path] using DRIVE_ROOT = '/content/drive/MyDrive/Ahsan/16. Review SynCellNet work'
  DATASET = PBMC
[1/4] Locating checkpoint:
      /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/scVI Approach/PBMC/checkpoints/best_model
      found: ['best_model.index', 'best_model.data-00000-of-00001']
[2/4] Restoring checkpoint…
[3/4] Weight change from restore:
      decoder.scale_h    mean|Δ| = 4.761e-02  changed=True
      z_encoder.mean_h   mean|Δ| = 1.251e-01  changed=True
      log_theta          mean|Δ| = 1.121e+00  changed=True
      ✓ all watched weights changed → load took effect
[4/4] recon vs real per-gene-mean correlation: 0.6347
      ✓ reconstruction tracks the real data
ALL CHECKS PASSED — model is on the trained checkpoint.

  b: (1186, 1600)  sparsity 99.7%  ->  /content/drive/MyDrive/Ahsan/16. Review SynCellNet work/Dataset/Synthetic PBMC dataset/scVI/PBMC/scvi_synthetic_b_expression_v3.csv
  mono: (1186, 1600)  sparsity 99.1%  ->  /content/drive/MyDrive/Ahsan/16. Revie